### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os, requests
r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}"}
)
for m in r.json()["data"]:
    print(m["id"])

canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
groq/compound-mini
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
groq/compound
openai/gpt-oss-20b
qwen/qwen3.6-27b
whisper-large-v3-turbo
openai/gpt-oss-120b
whisper-large-v3


In [7]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the same way humans use language; they are exceptional vocal mimics. Their ability to copy human speech (and other sounds) stems from a combination of anatomy, brain structure, and social behavior:\n\n| Factor | How it helps parrots mimic speech |\n|--------|-----------------------------------|\n| **Vocal anatomy** | Parrots have a highly flexible syrinx (the bird equivalent of a voice box) and a muscular, movable tongue and beak. This lets them produce a wide range of pitches and timbres, including the consonant‑vowel combinations that make up human words. |\n| **Auditory learning** | Their ears are tuned to pick up subtle variations in sound. Young parrots learn by listening to the calls of their parents and flock mates, and they apply the same learning mechanisms to any sounds they hear repeatedly in their environment. |\n| **Social nature** | In the wild, parrots live in tight‑knit flocks where vocal communication maintains group cohesion,

In [8]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [9]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks for weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_3b0f542e-f56d-4cad-8b08-cda33a025993', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 127, 'total_tokens': 167, 'completion_time': 0.083249368, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.004812774, 'prompt_tokens_details': None, 'queue_time': 0.158440046, 'total_time': 0.088062142}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a00f22-86b9-7782-a103-8768a2ee7f25-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_3b0f542e-f56d-4cad-8b08-cda33a025993', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_token

In [10]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is currently sunny. Enjoy your day!


In [11]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_f65bd23b-8849-4cdf-9e60-2fa14d7aa991', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 126, 'total_tokens': 166, 'completion_time': 0.083612231, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.005038923, 'prompt_tokens_details': None, 'queue_time': 0.315099015, 'total_time': 0.088651154}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9d1f936695', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00f25-3588-7693-b2d8-c43ec24930de-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_f65bd23b-8849-4cdf-9e60-2fa14d7aa